## ResNet-50 Transfer Learning — Task 1 (ver1_0_2)
Colab A100-aligned CNN baseline for **Normal** vs **Pneumonia**.
Training recipe kept comparable to prior ResNet runs; environment/dataset/seed aligned.

---


### Quick Setup Instructions
* Runtime → Change runtime type → **A100 GPU** (high-memory if available).
* Run install once, then proceed.


In [ ]:
# Lightweight Colab deps for CNN baseline (no Unsloth)
!pip -q install datasets scikit-learn seaborn tqdm huggingface_hub


In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torchvision import models, transforms
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pandas as pd
import re
from tqdm import tqdm

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DATASET_ID = "Henryayay/POLCOVID_task1_70_10_20"
BATCH_SIZE = 32
LEARNING_RATE = 1e-5
NUM_EPOCHS = 20
PATIENCE = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_MAP = {0: "Normal", 1: "Pneumonia"}
NAME_TO_ID = {"Normal": 0, "Pneumonia": 1}
TASK_ID = "Task1_ResNet"
RESULTS_PROJECT_ROOT = Path(
    r"C:\Users\Henryayay\OneDrive - The Hong Kong Polytechnic University\Desktop\AI xray classification project\Results"
)
LOCAL_RESULTS_ROOT = RESULTS_PROJECT_ROOT / "CNN" / "Task1"

print("Device:", DEVICE)
print("Dataset:", DATASET_ID)


In [ ]:
dataset = load_dataset(DATASET_ID)

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def collate_fn(batch, transform_func):
    images, labels = [], []
    for item in batch:
        img = item["image"].convert("RGB")
        images.append(transform_func(img))
        labels.append(item["label"])
    return torch.stack(images), torch.tensor(labels)

g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(
    dataset["train"], batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=lambda x: collate_fn(x, train_transforms),
    generator=g,
)
val_loader = DataLoader(
    dataset["validation"], batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=lambda x: collate_fn(x, test_transforms),
)
test_loader = DataLoader(
    dataset["test"], batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=lambda x: collate_fn(x, test_transforms),
)
print(len(dataset["train"]), len(dataset["validation"]), len(dataset["test"]))


In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

warmup_epochs = 2
warmup_scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
cosine_scheduler = CosineAnnealingLR(optimizer, T_max=(NUM_EPOCHS - warmup_epochs), eta_min=1e-7)
scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_epochs],
)


In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = -1.0
best_state = None
epochs_no_improve = 0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch} train"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    train_loss = running_loss / max(total, 1)
    train_acc = correct / max(total, 1)

    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_loss = running_loss / max(total, 1)
    val_acc = correct / max(total, 1)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    print(f"Epoch {epoch}: train_acc={train_acc:.4f} val_acc={val_acc:.4f} lr={optimizer.param_groups[0]['lr']:.2e}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

if best_state is not None:
    model.load_state_dict(best_state)


In [ ]:
# Test evaluation + results export
model.eval()
y_true_ids, y_pred_ids = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().tolist()
        y_pred_ids.extend(preds)
        y_true_ids.extend(labels.tolist())

y_true = [LABEL_MAP[i] for i in y_true_ids]
y_pred = [LABEL_MAP[i] for i in y_pred_ids]
acc = accuracy_score(y_true, y_pred) * 100
print(f"ResNet Task1 Accuracy: {acc:.2f}%")
print(classification_report(y_true, y_pred, labels=["Normal", "Pneumonia"]))
cm = confusion_matrix(y_true, y_pred, labels=["Normal", "Pneumonia"])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal", "Pneumonia"],
            yticklabels=["Normal", "Pneumonia"])
plt.title(f"ResNet Task1 Confusion Matrix")
plt.show()

def _slug(text):
    text = str(text).strip().replace(" ", "-")
    return re.sub(r"[^A-Za-z0-9._-]+", "_", text)

df = pd.DataFrame({
    "Index": range(len(y_true)),
    "True_Label_ID": y_true_ids,
    "Predicted_Label_ID": y_pred_ids,
    "True_Label_Name": y_true,
    "Predicted_Label_Name": y_pred,
})
filename = f"{_slug(TASK_ID)}_cnn_resnet50_acc{acc:.2f}pct.csv"
dest_dir = None
for candidate in (LOCAL_RESULTS_ROOT, Path.cwd() / "Results" / "CNN" / "Task1"):
    try:
        candidate.mkdir(parents=True, exist_ok=True)
        probe = candidate / ".write_test"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
        dest_dir = candidate
        break
    except Exception:
        continue
if dest_dir is None:
    dest_dir = Path.cwd() / "Results" / "CNN" / "Task1"
    dest_dir.mkdir(parents=True, exist_ok=True)
dest_path = dest_dir / filename
df.to_csv(dest_path, index=False)
print("Saved:", dest_path)
try:
    from google.colab import files
    if str(LOCAL_RESULTS_ROOT) not in str(dest_path):
        files.download(str(dest_path))
except Exception as exc:
    print("Download skipped:", exc)
